# 70b · Bước 1 — chuẩn bị dữ liệu cho bộ sinh

**Chạy CPU thuần.** Không nạp ảnh, không nạp model — chỉ đọc dự đoán val đã có sẵn của
ba thành viên. Vì vậy chạy **song song được** với `70a` đang chiếm GPU.

Ba việc, theo `docs/PLAN_SINH_ANH_IMG2IMG.md` Bước 1:

```
C2  chan doan LOI TAT HEAD  ->  thu tu uu tien thu nhanh P-xoa
C3  bang dong mac p(y_head | y_j = 1)  ->  prompt am
C4  chon anh goc: xep theo SO NHAN DUONG tang dan
```

⚠ Ghi vào `stage4_nb70b`, **không** dùng chung thư mục với `70a`: hai notebook có thể
chạy cùng lúc và `ensure_res()` của cả hai sẽ đua nhau `makedirs` — đúng cơ chế đã đẻ ra
17 thư mục trùng tên ở `68b`.

## C0 · Config — CPU, không nạp ảnh, không nạp model

In [ ]:
import os, json, glob, time, shutil, datetime, re
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

RES_LOC = '/content/nb70b'; os.makedirs(RES_LOC, exist_ok=True)
RES_DRV = os.path.join(OUT_DIR, 'stage4_nb70b')          # RIENG, khong dung chung voi 70a
_RES_OK = False


def ensure_res():
    """Tao DUNG MOT LAN moi session, xac minh bang GHI THAT + DOC LAI, roi khong bao gio
       goi makedirs nua. GIU lai file danh dau: Drive khong giu thu muc RONG dang tin
       qua FUSE (15/09 da mat mot thu muc y het kieu do)."""
    global _RES_OK
    if _RES_OK: return RES_DRV
    if not os.path.isdir(RES_DRV):
        os.makedirs(RES_DRV, exist_ok=True)
    _t = os.path.join(RES_DRV, '_tao_boi_70b.txt')
    with open(_t, 'w') as _f: _f.write('70b · Buoc 1. Dung xoa: thu muc rong se bien mat.\n')
    assert open(_t).read().startswith('70b'), f'{RES_DRV} ghi-doc lai khong duoc -> mount hong'
    _RES_OK = True
    print(f'thu muc ket qua: {RES_DRV}  (tao 1 lan, da xac minh)')
    return RES_DRV


PUT_FAIL = []
def put(name, fn):
    lp = os.path.join(RES_LOC, name); fn(lp)
    dp = os.path.join(RES_DRV, name)
    assert not os.path.exists(dp), (
        f'{name} DA CO tren Drive. Khong ghi de: qua FUSE thi lan ghi thu hai co the '
        'khong xuong dia ma van bao thanh cong. Ten file co dau thoi gian roi ma van '
        'trung thi la chay hai lan trong cung mot phut.')
    global _RES_OK
    _loi = ''
    for _lan in range(3):
        try:
            ensure_res(); shutil.copy(lp, dp)
            if open(dp, 'rb').read() != open(lp, 'rb').read():
                raise OSError('noi dung tren Drive KHAC ban local')
            return lp
        except Exception as e:
            _loi = f'{type(e).__name__}: {e}'
            if isinstance(e, FileNotFoundError): _RES_OK = False
            time.sleep(2 * (_lan + 1))
    PUT_FAIL.append(name); print(f'  !! sao {name} len Drive THAT BAI sau 3 lan ({_loi})')
    return lp


ensure_res()

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)

# CHI CO BA TAP. Khong con lat chon / lat khop / tap con nao khac.
assert not (set(SUB[TR]) & set(SUB[VA])), 'benh nhan train ro ri sang VAL'
assert not (set(SUB[TR]) & set(SUB[TE])), 'benh nhan train ro ri sang TEST'
I_TR, I_VA, I_TE = (np.where(m)[0] for m in (TR, VA, TE))
assert len(I_TR) + len(I_VA) + len(I_TE) == len(ORDER), 'ba tap khong phu het'

_prev = YM[TR].mean(0)
TIER = {'head':   np.where(_prev > 0.10)[0],
        'medium': np.where((_prev >= 0.01) & (_prev <= 0.10))[0],
        'tail':   np.where(_prev < 0.01)[0]}
HEAD = TIER['head']
THIEU_SO = np.concatenate([TIER['medium'], TIER['tail']])      # 31 nhan
Y_VA = YM[I_VA]
print(f'{len(ORDER):,} anh = train {len(I_TR):,} + val {len(I_VA):,} + test {len(I_TE):,}')
print(f'TIER CHINH THUC {dict((k, len(v)) for k, v in TIER.items())} '
      f'| nhan thieu so: {len(THIEU_SO)}')
print('so duong tinh tren TRAIN cua nhan thieu so: '
      f'{int(YM[TR][:, THIEU_SO].sum(0).min()):,} .. {int(YM[TR][:, THIEU_SO].sum(0).max()):,}')


## C1 · Nạp dự đoán val của ba thành viên

Đọc `p_{arm}_s*_ep*.npy` — dự đoán trên **val** đã lưu sẵn từ các lượt huấn luyện.
Không tính lại gì.

⚠ Duyệt cả thư mục **trùng tên** (`r2a`, `r2a (1)`, …). Drive cho phép trùng tên, và bỏ
qua bước này thì chỉ thấy một phần file — đúng lỗi đã làm `68b` mất nửa số seed.

In [ ]:
# ==== nap du doan VAL cua ba thanh vien ====
THANH_VIEN = [
    ('ong',     'stage4_nb64',  'r2a',        'tpb'),        # duong ong RAD-DINO
    ('cnn518',  'stage4_nb64b', 'r2a',        'cnn_full'),
    ('cnn1022', 'stage4_nb68b', 'r1022_E8',   'cnn_full'),
]


def _thu_muc(goc, run):
    """Drive cho phep thu muc TRUNG TEN -> phai duyet HET."""
    out = []
    for x in sorted(os.listdir(OUT_DIR)):
        if not (x == goc or x.startswith(goc + ' (')): continue
        b = os.path.join(OUT_DIR, x)
        if not run:
            out.append(b); continue
        for y in sorted(os.listdir(b)):
            if (y == run or y.startswith(run + ' (')) and os.path.isdir(os.path.join(b, y)):
                out.append(os.path.join(b, y))
    return out


def _doc(p):
    """Copy ve /content roi doc. Doc THANG tren mount FUSE la nguon goc
       `OSError 107 Transport endpoint is not connected` (da dinh nhieu lan)."""
    lp = os.path.join(RES_LOC, '_tmp_' + p.replace(os.sep, '_').replace(':', '_').lstrip('_'))
    if not (os.path.exists(lp) and os.path.getsize(lp) == os.path.getsize(p)):
        shutil.copy(p, lp)
        assert os.path.getsize(lp) == os.path.getsize(p), f'copy NGAN tu Drive: {p}'
    return lp


PV = {}
for nhan, goc, run, arm in THANH_VIEN:
    uq = {}
    for d in _thu_muc(goc, run):
        for f in sorted(glob.glob(os.path.join(d, f'p_{arm}_s*_ep*.npy'))):
            uq.setdefault(os.path.basename(f), f)      # trung ten -> dem MOT lan
    assert uq, (f'{nhan}: khong thay p_{arm}_s*_ep*.npy trong {goc}/{run}. '
                'Kiem thu muc trung ten tren Drive.')
    Q = np.mean([np.load(_doc(f)).astype('float32') for f in uq.values()], 0)
    assert Q.shape == (len(I_VA), C), f'{nhan}: hinh dang {Q.shape}, mong doi {(len(I_VA), C)}'
    assert np.isfinite(Q).all(), f'{nhan}: co NaN/Inf'
    PV[nhan] = Q
    print(f'  {nhan:<10}{len(uq):>4} file  -> {Q.shape}')


def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o


print()
for nhan, Q in PV.items():
    a = ap_cols(Y_VA, Q)
    print(f'  {nhan:<10}macro {np.nanmean(a):.4f} | '
          + ' '.join(f'{t} {np.nanmean(a[i]):.4f}' for t, i in TIER.items()))


## C2 · Chẩn đoán lối tắt head

Giả thuyết trung tâm của cả hướng: *mô hình nhận ra nhãn thiểu số qua nhãn head đồng mắc
chứ không qua đặc trưng thật*. Đo được mà không cần GPU: với mỗi nhãn `j`, hồi quy
logistic từ **9 dự đoán head của chính mô hình** ra `j`, cross-fit theo bệnh nhân, rồi so
`AP_head_only(j)` với `AP_model(j)`.

```
ty so ~ 1   ->  chi bao: j co the doc duoc qua head  ->  uu tien thu P-xoa truoc
ty so ~ 0   ->  chi bao nguoc lai
```

⚠ **Cả hai chiều đều chưa chứng minh gì.** Tỉ số cao không chứng minh mô hình *dùng* lối
tắt — `j` có thể vừa đồng mắc thật với head vừa được đọc đúng qua đặc trưng riêng. Tỉ số
thấp cũng không chứng minh mô hình đọc đặc trưng thật. Đây **chỉ là thứ tự ưu tiên**.

Chạy trên **cả ba thành viên** riêng rẽ. Ba cái đồng ý thì kết luận chắc hơn; lệch nhau
thì chính sự lệch đó là thông tin.

In [ ]:
# ==== chan doan loi tat head ====
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

SUB_VA = SUB[I_VA]
_gkf = GroupKFold(n_splits=5)
TYSO = {}

for nhan, Q in PV.items():
    X = np.log(np.clip(Q[:, HEAD], 1e-6, 1 - 1e-6) /
               (1 - np.clip(Q[:, HEAD], 1e-6, 1 - 1e-6)))     # logit cua 9 du doan head
    ap_m = ap_cols(Y_VA, Q)
    ty = np.full(C, np.nan)
    for j in THIEU_SO:
        y = Y_VA[:, j]
        if y.sum() < 10: continue
        oof = np.zeros(len(y), 'float32')
        for tr, te in _gkf.split(X, y, groups=SUB_VA):
            if y[tr].sum() < 2: continue
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit(X[tr], y[tr])
            oof[te] = lr.predict_proba(X[te])[:, 1]
        ap_h = ap_cols(y[:, None], oof[:, None])[0]
        ty[j] = ap_h / max(ap_m[j], 1e-9)
    TYSO[nhan] = ty
    print(f'  {nhan}: xong')

_tier_cua = {j: t for t, ii in TIER.items() for j in ii}
_npos = YM[TR].sum(0).astype(int)
print('\n' + '=' * 84)
print(f'{"nhan":<26}{"tier":<8}{"n+ train":>9}  ' + '  '.join(f'{n:>13}' for n in PV))
print('-' * 84)
for j in sorted(THIEU_SO, key=lambda x: -np.nanmean([TYSO[n][x] for n in PV])):
    print(f'{LAB40[j][:25]:<26}{_tier_cua[j][:6]:<8}{_npos[j]:>9,}  '
          + '  '.join(f'{TYSO[n][j]:>13.2f}' if np.isfinite(TYSO[n][j]) else f'{"—":>13}'
                      for n in PV))
print('-' * 84)
print('ty so = AP(chi tu 9 du doan head) / AP(ca mo hinh). Cao = nhan do doc duoc phan lon')
print('tu head. CHUA chung minh mo hinh DUNG loi tat — chi la thu tu uu tien thu P-xoa.')

_tb = np.array([np.nanmean([TYSO[n][j] for n in PV]) for j in THIEU_SO])
print(f'\nty so trung binh ba thanh vien: min {np.nanmin(_tb):.2f} | '
      f'trung vi {np.nanmedian(_tb):.2f} | max {np.nanmax(_tb):.2f}')
_cao = [LAB40[j] for j, v in zip(THIEU_SO, _tb) if np.isfinite(v) and v > np.nanmedian(_tb)]
print(f'{len(_cao)}/{len(THIEU_SO)} nhan tren trung vi -> uu tien thu P-xoa truoc o nhom nay')


## C3 · Bảng đồng mắc → prompt âm, và chọn ảnh gốc

**Prompt âm** lấy ba nhãn head đồng mắc nhiều nhất với `j` trên train. Đây mới là cơ chế
chuyên biệt hoá thật sự: ảnh gốc đã dương với `j` nên prompt dương chỉ giữ `j` khỏi bị
nhiễu làm mất, còn prompt âm **đẩy ảnh ra xa biểu hiện head** trong lúc khử nhiễu.

**Chọn ảnh gốc**: xếp ảnh dương của `j` theo **số nhãn dương tăng dần**. Ảnh mà `j` gần
như là nhãn duy nhất thì prompt âm phải xoá ít hơn ⇒ ảnh sinh sạch hơn, và EHR kế thừa
ít mâu thuẫn hơn.

In [ ]:
# ==== bang dong mac + chon anh goc ====
Y_TR = YM[TR]
_npos_tr = Y_TR.sum(0)
SO_NHAN = Y_TR.sum(1)                      # so nhan duong cua TUNG anh train
print(f'so nhan duong moi anh train: trung binh {SO_NHAN.mean():.2f} | '
      f'trung vi {int(np.median(SO_NHAN))} | max {int(SO_NHAN.max())}')

PROMPT, NGUON = {}, {}
for j in THIEU_SO:
    m = Y_TR[:, j] == 1
    n_j = int(m.sum())
    p_head = Y_TR[m][:, HEAD].mean(0)                        # p(y_head | y_j = 1)
    top = HEAD[np.argsort(-p_head)[:3]]
    PROMPT[LAB40[j]] = dict(
        duong=f'chest radiograph, {LAB40[j]}',
        am=', '.join(LAB40[h] for h in top),
        p_dong_mac=[round(float(p_head[list(HEAD).index(h)]), 3) for h in top])
    # anh goc: xep theo SO NHAN DUONG tang dan, roi theo thu tu on dinh
    idx_tr = I_TR[m]
    NGUON[LAB40[j]] = idx_tr[np.argsort(SO_NHAN[m], kind='stable')].tolist()

print('\n' + '=' * 96)
print(f'{"nhan":<26}{"tier":<8}{"n+":>7}{"nhan/anh":>10}  prompt am (ba head dong mac nhat)')
print('-' * 96)
for j in sorted(THIEU_SO, key=lambda x: _npos_tr[x]):
    nm = LAB40[j]; m = Y_TR[:, j] == 1
    print(f'{nm[:25]:<26}{_tier_cua[j][:6]:<8}{int(_npos_tr[j]):>7,}'
          f'{SO_NHAN[m].mean():>10.2f}  {PROMPT[nm]["am"][:44]}')
print('-' * 96)

# bao nhieu anh goc gan nhu DON NHAN (<=2 nhan duong, tuc j + nhieu nhat mot nhan khac)
print(f'\n{"nhan":<26}{"n+":>7}{"<=2 nhan":>10}{"ti le":>8}   5 anh goc sach nhat (so nhan)')
print('-' * 96)
for j in sorted(THIEU_SO, key=lambda x: _npos_tr[x]):
    nm = LAB40[j]; ds = NGUON[nm]
    _s = SO_NHAN[Y_TR[:, j] == 1]
    _n2 = int((_s <= 2).sum())
    _dau = ', '.join(str(int(SO_NHAN[np.where(I_TR == d)[0][0]])) for d in ds[:5])
    print(f'{nm[:25]:<26}{len(ds):>7,}{_n2:>10,}{_n2/max(len(ds),1):>8.1%}   {_dau}')
print('-' * 96)
print('cot "<=2 nhan" = so anh goc ma j gan nhu la nhan duy nhat. Cang nhieu cang tot:')
print('prompt am phai xoa it hon -> anh sinh sach hon, EHR ke thua it mau thuan hon.')

# ---------- luu ----------
_kq = dict(ngay=datetime.datetime.now().isoformat(timespec='seconds'),
           nhan=LAB40, tier={t: [int(x) for x in i] for t, i in TIER.items()},
           thieu_so=[int(x) for x in THIEU_SO],
           prompt=PROMPT,
           ty_so_loi_tat={n: [None if not np.isfinite(v) else round(float(v), 4)
                              for v in TYSO[n]] for n in TYSO},
           n_nguon={k: len(v) for k, v in NGUON.items()})
_T = f'buoc1_{datetime.datetime.now():%Y%m%d_%H%M}'
put(f'{_T}.json', lambda p: json.dump(_kq, open(p, 'w'), ensure_ascii=False, indent=1))
put(f'{_T}_nguon.json', lambda p: json.dump(
    {k: [int(x) for x in v] for k, v in NGUON.items()}, open(p, 'w')))
print(f'\nda luu {_T}.json + _nguon.json -> {RES_DRV.replace(OUT_DIR, "...")}')

if PUT_FAIL:
    raise AssertionError(
        f'{len(PUT_FAIL)} file KHONG len duoc Drive: {PUT_FAIL[:4]}\n'
        f'  Hien vat CON o {RES_LOC}. Cuu: !cp -r {RES_LOC} /content/drive/MyDrive/_cuu70b')
print('cong Drive: tat ca file da len')
